In [58]:
import pandas as pd 
import datetime
JOURNAL_CODES = ['HA', 'CA', 'BC', 'AN']


In [ ]:
cleaned_rows = []
current_fournisseur_number = None
current_fournisseur_name = None
start_date = datetime.datetime.strptime('2025-10-01', '%Y-%m-%d').date() 
end_date = datetime.datetime.strptime('2025-12-31', '%Y-%m-%d').date() 
start_year  = datetime.datetime.strptime('2025-01-01', '%Y-%m-%d').date() 
start_date_1 = datetime.datetime.strptime('2025-09-30', '%Y-%m-%d').date() 

df_raw = pd.read_excel('GRAND LIVRE FOURNISSEURS 2025.xlsx', header=None) 
for i, row in df_raw.iterrows():
        first_cell = str(row[0]).strip()

        if first_cell.startswith('4411'):
            current_fournisseur_number = first_cell.split()[0]
            current_fournisseur_name = ' '.join(first_cell.split()[1:])

        elif str(row[1]).strip() in JOURNAL_CODES:
            cleaned_rows.append({
                'Num fournisseur': current_fournisseur_number,
                'Nom fournisseur': current_fournisseur_name,
                'journal': row[1],
                'date': row[2].date() if pd.notna(row[2]) else None,
                'N° de pièce': row[3],
                'libelle': row[4],
                'montant_debit': row[8],
                'montant_credit': row[10],
                'lettrage': row[9]
            })

df_clean = pd.DataFrame(cleaned_rows)

# Facture trimestre paye / Non paye

In [ ]:



factures_trimester = df_clean[
    (df_clean['journal'].isin(['HA', 'AN'])) &
    (df_clean['montant_credit'].notna()) &
    (df_clean['montant_credit'] > 0)
    &( df_clean['date'].between(start_date, end_date) )
]
paiements_trimester = df_clean[
    (df_clean['journal'].isin(['BC', 'CA','AN'])) &
    (
        (df_clean['montant_debit'].notna()) |
        (df_clean['montant_credit'] < 0)
    )&
    ( df_clean['date'].between(start_date, end_date)  )

]
result = pd.merge(
        factures_trimester,
        paiements_trimester[['Num fournisseur', 'lettrage', 'montant_debit', 'date']],
        on=['Num fournisseur', 'lettrage'],
        how='left',
        suffixes=('_facture', '_paiement')
    )


# excel = result.to_excel('output.xlsx', index=None)



,Num fournisseur,Nom fournisseur,journal,date_facture,N° de pièce,libelle,montant_debit_facture,montant_credit,lettrage,montant_debit_paiement,date_paiement
0,44110000,Fournisseurs,HA,2025-10-10,25020470,TIGRA DISTRIBUTION (DACIA),NaN,830.50,AFC,830.50,2025-10-13
1,44110000,Fournisseurs,HA,2025-10-22,2510-3144,NAB (BATTERIE MERCEDES),NaN,2450.00,AFB,2450.00,2025-10-24
2,44110000,Fournisseurs,HA,2025-10-24,2510-212,CENTRE AUTO FEU VERT (MERCEDES...,NaN,562.00,AFD,562.00,2025-10-28
3,44110000,Fournisseurs,HA,2025-12-02,3276,STAR TUNING (FORD),NaN,350.00,AFE,350.00,2025-12-09
4,44110000,Fournisseurs,HA,2025-12-09,199 / 12-25,KAMY STEEL (SERURE. BUREAU SEM...,NaN,350.00,AFF,350.00,2025-12-18
...,...,...,...,...,...,...,...,...,...,...,...
239,44113039,TURBOCARGO MAROC (TCM),HA,2025-11-20,2025/FV/0396,T C M / KOPIMASK,NaN,19177.10,AAF,19177.10,2025-11-26
240,44113039,TURBOCARGO MAROC (TCM),HA,2025-12-10,2025/FV/0429,TCM / PRINTER WORLD,NaN,7774.10,AAH,7774.10,2025-12-15
241,44113039,TURBOCARGO MAROC (TCM),HA,2025-12-10,2025/FV/0428,TCM / MINERVA,NaN,16623.56,AAI,16623.56,2025-12-15
242,44113039,TURBOCARGO MAROC (TCM),HA,2025-12-18,2025/FV/0440,T C M / POLICROM,NaN,7535.26,AAG,7535.26,2025-12-22


# Past invoice paid in trimester

In [68]:

factures_avant_trimester = df_clean[
    (df_clean['journal'].isin(['HA', 'AN'])) &
    (df_clean['montant_credit'].notna()) &
    (df_clean['montant_credit'] > 0)
    &( df_clean['date'].between(start_year, end_date)  )
]
paiements_trimester = df_clean[
    (df_clean['journal'].isin(['BC', 'CA','AN'])) &
    (
    (df_clean['montant_debit'].notna()) |
        (df_clean['montant_credit'] < 0)
    )&
    ( df_clean['date'].between(start_date, end_date)  )
]

join = pd.merge(
        factures_avant_trimester,
        paiements_trimester[['Num fournisseur', 'lettrage', 'montant_debit', 'date']],
        on=['Num fournisseur', 'lettrage'],
        how='left',
        suffixes=('_facture', '_paiement')
    )
paiements_trimester
# result = join[join['montant_debit_paiement'].notna() ]

# excel = result.to_excel('output.xlsx', index=None)



,Num fournisseur,Nom fournisseur,journal,date,N° de pièce,libelle,montant_debit,montant_credit,lettrage
35,44110000,Fournisseurs,CA,2025-10-03,19760,REGT.FACT/ CORDERIE DE LA KOUT...,500.00,NaN,AFA
37,44110000,Fournisseurs,CA,2025-10-13,25020470,REGT.FACT/ TIGRA DISTRIBUTION ...,830.50,NaN,AFC
39,44110000,Fournisseurs,BC,2025-10-24,2510-3144,REGT.CHQ°6509114 / NBA,2450.00,NaN,AFB
41,44110000,Fournisseurs,CA,2025-10-28,2510-212,REGT.FACT/ FEU VERT,562.00,NaN,AFD
43,44110000,Fournisseurs,CA,2025-12-09,3276,REGT.FACT/ STAR TUNING,350.00,NaN,AFE
...,...,...,...,...,...,...,...,...,...
1131,44113039,TURBOCARGO MAROC (TCM),BC,2025-11-26,2025 /FV/396,REGT.CHQ°65091/ T C M,19177.10,NaN,AAF
1134,44113039,TURBOCARGO MAROC (TCM),BC,2025-12-15,2025/FV/0429,REGT.CHQ°6509142/ T C M,7774.10,NaN,AAH
1135,44113039,TURBOCARGO MAROC (TCM),BC,2025-12-15,2025/FV/0428,REGT.CHQ°6509144 / T C M,16623.56,NaN,AAI
1137,44113039,TURBOCARGO MAROC (TCM),BC,2025-12-22,2025/FV/0440,REGT.CHQ°6509148/ T C M,7535.26,NaN,AAG


# Past invoice not paid

In [ ]:

factures_trimester = df_clean[
    (df_clean['journal'].isin(['HA', 'AN'])) &
    (df_clean['montant_credit'].notna()) &
    (df_clean['montant_credit'] > 0)
    &( df_clean['date'].between(start_year, end_date)  )
]
paiements_year = df_clean[
    (df_clean['journal'].isin(['BC', 'CA','AN'])) &
    (
        (df_clean['montant_debit'].notna()) |
        (df_clean['montant_credit'] < 0)
    )&
    ( df_clean['date'].between(start_date, end_date)  )
]

result = pd.merge(
        factures_trimester,
        paiements_year[['Num fournisseur', 'lettrage', 'montant_debit', 'date']],
        on=['Num fournisseur', 'lettrage'],
        how='left',
        suffixes=('_facture', '_paiement')
    )


# excel = result.to_excel('output.xlsx', index=None)

